# EasyRAG Repo Loading Basics

This notebook focuses on the data-loading boundary in EasyRAG. It shows how a small repository-shaped corpus becomes canonical `Document` objects, how metadata is attached before indexing, and how chunk previews reveal problems early.

## What you will do

- create a tiny repository-shaped corpus
- load repository documents through the canonical EasyRAG entry point
- inspect the metadata that later powers grounded citations
- preview chunk boundaries before any persistent workspace is built

## Why this notebook stays before index building

Indexing quality starts with input quality. If you do not know what was loaded, how the paths were normalized, or where chunk boundaries look suspicious, later retrieval results will be harder to interpret.


## Step 1: Import the indexing and query helpers

We will use the public `EasyRAG` API for loading and querying, and a small set of indexing helpers for chunk inspection and workspace rebuilds. `run_sync` is still useful because the orchestrator lifecycle is async even inside a notebook.


In [ ]:
import json
import tempfile
from pathlib import Path

from easyrag import EasyRAG
from easyrag.rag.indexing import ChunkingConfig, chunk_documents


This cell should only import symbols. The important thing to notice is the separation of roles: `EasyRAG` is the high-level entry point, while `chunk_documents()` and `rebuild_document_index()` expose the indexing mechanics more directly.


## Step 2: Create a tiny repository-shaped corpus

Instead of indexing the full EasyRAG repository, we create a small temporary repo with a `README.md` and a few files under `docs/`. That keeps the example deterministic and makes it obvious why each document was loaded.


In [ ]:
repo_temp_dir = tempfile.TemporaryDirectory()
repo_root = Path(repo_temp_dir.name) / "demo_repo"
docs_dir = repo_root / "docs"
docs_dir.mkdir(parents=True)

(repo_root / "README.md").write_text(
    "# Demo Repo\nThis repository explains EasyRAG indexing.\n",
    encoding="utf-8",
)
(docs_dir / "architecture.md").write_text(
    "# Architecture\nEasyRAG organizes indexing as loading, chunking, and retrieval.\n## Chunking\nStructured chunks keep section boundaries.\n",
    encoding="utf-8",
)
(docs_dir / "notes.txt").write_text(
    "Semantic chunking works well for plain text notes about embeddings and retrieval.\n",
    encoding="utf-8",
)
(docs_dir / "operations.md").write_text(
    "# Operations\nThe build script can rebuild, update, or delete index entries.\n",
    encoding="utf-8",
)

print(json.dumps(sorted(str(path.relative_to(repo_root)) for path in repo_root.rglob("*") if path.is_file()), indent=2))


You should see a small file list rooted at `demo_repo/`. This is the corpus we will index. Because the directory layout looks like a repository, the same loader logic used on real docs can operate on it without any special casing.


## Step 3: Load repository documents

`EasyRAG.load_repo_documents()` is the teaching-friendly entry point for repository discovery. It applies the repo loader rules and returns canonical `Document` objects with metadata such as title, path, and `doc_id`.


In [ ]:
documents = EasyRAG.load_repo_documents(repo_root)

document_preview = [
    {
        "title": str(document.metadata.get("title")),
        "relative_path": str(document.metadata.get("relative_path")),
        "doc_id": str(document.metadata.get("doc_id")),
        "source_type": str(document.metadata.get("source_type")),
    }
    for document in documents
]

print(json.dumps(document_preview, indent=2))


You should see one record per loaded source file. The key observation is that indexing does not begin with anonymous text blobs. It begins with documents that already carry stable metadata, and that metadata will later show up in citations and storage records.


## Step 4: Inspect chunk boundaries before building the workspace

Chunking is where raw documents become retrieval units. We inspect chunks before writing any index so you can see which strategy EasyRAG selected and what metadata each chunk carries.


In [ ]:
chunks = chunk_documents(documents, config=ChunkingConfig())

chunk_preview = [
    {
        "doc_id": str(chunk.metadata.get("doc_id")),
        "title": str(chunk.metadata.get("title")),
        "chunk_id": int(chunk.metadata.get("chunk_id", 0)),
        "chunk_strategy": str(chunk.metadata.get("chunk_strategy")),
        "overlap_policy": str(chunk.metadata.get("overlap_policy")),
        "snippet": chunk.page_content[:90],
    }
    for chunk in chunks
]

print(json.dumps(chunk_preview, indent=2))


This output should show a mix of strategies. Markdown files usually become `structured` chunks, while plain text files often fall back to `sliding_window` when no semantic embedding function is available. That is a useful reminder that chunking is adaptive, not one-size-fits-all.


## Step 5: Clean up the temporary repository

We are done with the loading-side walkthrough. Cleaning up the temporary repo keeps the notebook self-contained and makes it obvious that we have not built a persistent EasyRAG workspace yet.

In [ ]:
repo_temp_dir.cleanup()

After cleanup, the temporary repository is gone. That is expected. This notebook is about getting source material into canonical `Document` objects and inspecting the first chunk boundary decisions.

## Next steps

- Continue with [03_07_build_index_pipeline.ipynb](../03_indexing/03_07_build_index_pipeline.ipynb) to turn the same kind of corpus into a persistent workspace.
- Compare this repository-shaped path with [02_02_manual_document_preparation.ipynb](02_02_manual_document_preparation.ipynb) if your inputs start as in-memory texts instead of files.
- Read [02-data-loading-overview.md](../../docs/02-data-loading-overview.md) for the stage-level explanation behind these two loading paths.